<a href="https://colab.research.google.com/github/smduarte/spbd-2627/blob/main/docs/labs/lab3/SPBD_Labs_mapreduce1_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 - MapReduce + MRJob Library - Solutions

[MrJob](https://mrjob.readthedocs.io/en/latest/) can be used to write MapReduce jobs and run them on several platforms.

Some key advantages:
+ Write **multi-step** MapReduce jobs in pure Python;
+ Test on your **local machine**;
+ Deploy jobs in several cloud plataforms of several vendors.


 <br>
 Note: The lastest version of MrJob requires python 3.12 or older.

In [ ]:
#@title Install Python3.12 + MrJob
!apt-get update -y -qq > /dev/null 2>&1
!apt-get install python3.12 python3-pip -qq -y
!python3.12 -m pip install mrjob --break-system-packages --quiet
!pip install mrjob --quiet


##1. Word Frequency

Create a notebook program that counts the frequency of each word in a text file (for example, the novel "Os Maias").

a) No sorting required: words and their frequencies can appear in any order.

b) Sorted by word, in ascending alphabetical order.

c) Sorted by frequency, in descending order (most frequent words first).


In [ ]:
#@title Download Dataset
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

### a) Word Frequency - No sorting required

The mapper emits `(word, 1)` for every word in a line, and the reducer sums the final count per word.

In [ ]:
%%writefile word_freq_1a.py

import re
from mrjob.job import MRJob
from mrjob.step import MRStep
class MRWordFreq1a(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r"[^\W_]+", line.lower()):
            yield word, 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordFreq1a.run()


In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1a -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

In [ ]:
%%writefile word_freq_1a_comb.py
#@title Using the combiner

import re
from mrjob.job import MRJob

class MRWordFreq1aComb(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r"[^\W_]+", line.lower()):
            yield word, 1

    def combiner(self, word, counts):
        yield word, sum(counts)

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordFreq1aComb.run()


In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1a_comb -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

### b) Sorted by word (ascending)

In [ ]:
%%writefile word_freq_1b.py
import re
from mrjob.job import MRJob, MRStep

class MRWordFreq1b(MRJob):

    # ---------- Step 1: count words ----------
    def mapper_count(self, _, line):
        for word in re.findall(r"[^\W_]+", line.lower()):
            yield word, 1

    def combiner_count(self, word, counts):      # local pre-aggregation
        yield word, sum(counts)

    def reducer_count(self, word, counts):
        yield word, sum(counts)

    # ---------- Step 2: global sort ----------
    def mapper_collect(self, word, count):
        yield None, (word, count)

    def reducer_sort(self, _, pairs):
        for word, count in sorted(pairs):
            yield word, count

    def steps(self):
        return [MRStep(mapper=self.mapper_count, combiner=self.combiner_count,reducer=self.reducer_count),
                MRStep(mapper=self.mapper_collect, reducer=self.reducer_sort)]

if __name__ == "__main__":
    MRWordFreq1b.run()

In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1b -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

### c) Sorted by frequency (descending)

In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m word_freq_1c -r local --output-dir results --cleanup NONE os_maias.txt
!cat results/* | head -10

In [ ]:
%%file word_freq_1c.py

import re
from mrjob.job import MRJob, MRStep

MAX_FREQ = 1000000
class MRWordCountFrequency1c(MRJob):

  def mapper_frq(self, _, line):
    for word in re.findall(r"[^\W_]+", line.lower()):
      yield word, 1

  def combiner_frq(self, word, counts ):
      yield word, sum( counts )

  def reducer_frq(self, word, counts):
      yield word, sum( counts )

  def mapper_sort(self, word, counts):
      yield "{:06}".format(MAX_FREQ-int(counts)), word

  def reducer_sort(self, frequency, words):
    for word in words:
      yield word, MAX_FREQ - int(frequency)

  def steps(self):
    return [ MRStep(mapper=self.mapper_frq, combiner=self.combiner_frq, reducer=self.reducer_frq),
            MRStep(mapper=self.mapper_sort, reducer=self.reducer_sort)
    ]

if __name__ == '__main__':
    MRWordCountFrequency1c.run()

## 2. Weblog Analysis

Consider a set of log files captured during a DDoS (Distributed Denial of Service) attack, containing web access information recorded by the server during the attack.

Create a python notebook that processes these web log entries to:

a) Count the number of unique IP addresses involved in the DDoS attack.

b) For each 10-second interval, calculate the following metrics: [number of requests, average execution time, maximum execution time, minimum execution time].

c) Build an inverted index that maps each 10-second interval to a list of unique IP addresses that made requests (grouped by URL/Path).

The log files contain tab-separated text lines formatted as follows:


The log files contain text lines as shown below, with blank as the separator:

date |IP_source | status_code | operation | URL/Path | execution time |
-|-|-|-|-|-
timestamp  | string | int | string | string| float |
2016-12-06T08:58:35.318+0000|37.139.9.11|404|GET|/codemove/TTCENCUFMH3C|0.026

In [ ]:
#@title Download Dataset
!wget -q -O weblog.txt https://www.dropbox.com/s/0r8902uj9yum7dg/web.log?dl=0
!head -3 weblog.txt

In [ ]:
#@title Exercise 2a) - Count Unique IPs
%%file weblog_stats_2a.py

from mrjob.job import MRJob, MRStep

class MRWebLogStats2a(MRJob):

  def mapper1(self, _, line):
      parts = line.strip().split(' ')
      if len(parts) == 6:   # make sure the line has at least date + IP
            ip = parts[1]
            yield ip, None

  def combiner1(self, ip, _):
      yield ip, None

  def reducer1(self, ip, _):
      yield None, 1

  def reducer2(self, _, ips):
      yield "Unique Ips", sum(ips)

  def steps(self):
    return [ MRStep(mapper=self.mapper1, combiner=self.combiner1, reducer=self.reducer1),
             MRStep(reducer=self.reducer2)]

if __name__ == '__main__':
    MRWebLogStats2a.run()

In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m weblog_stats_2a -r local --output-dir results --cleanup NONE weblog.txt
!cat results/*

In [ ]:
#@title Exercise 2b) 10-second interval stats (version 1)
%%file weblog_stats_2b_v1.py

from mrjob.job import MRJob, MRStep
from statistics import mean

class MRWebLogStats2b_V1(MRJob):

  def mapper(self, _, line):
      parts = line.strip().split(' ')
      if len(parts) == 6:   # make sure the line is complete
            timestamp = parts[0]
            execution_time = float(parts[5])

            time_interval_10s = timestamp[0:18]
            yield time_interval_10s, execution_time

  def reducer(self, interval, execution_times):
      values = list(execution_times)
      yield interval, "count: {}, min: {}, max: {}, avg: {}".format(len(values), min(values), max(values), mean(values))

if __name__ == '__main__':
    MRWebLogStats2b_V1.run()

In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m weblog_stats_2b_v1 -r local --output-dir results --cleanup NONE weblog.txt
!cat results/*

In [ ]:
#@title Exercise 2b) 10-second interval stats (version 2)
%%file weblog_stats_2b_v2.py

from mrjob.job import MRJob, MRStep

class MRWebLogStats2b_V2(MRJob):

  def mapper(self, _, line):
      parts = line.split()
      if len(parts) == 6:   # make sure the line is complete
            timestamp = parts[0]
            execution_time = float(parts[5])

            time_interval_10s = timestamp[0:18]
            yield time_interval_10s, (1, execution_time, execution_time, execution_time)

  def combiner(self, interval, execution_times):
    values = list(execution_times)

    requests = sum([x[0] for x in values])
    min_time = min([x[1] for x in values])
    max_time = max([x[2] for x in values])
    sum_time = sum([x[3] for x in values])
    yield interval, (requests, min_time, max_time, sum_time)

  def reducer(self, interval, execution_times):
    values = list(execution_times)

    requests = sum([x[0] for x in values])
    min_time = min([x[1] for x in values])
    max_time = max([x[2] for x in values])
    sum_time = sum([x[3] for x in values])

    yield interval, "count: {}, min: {}, max: {}, avg: {}".format(requests, min_time, max_time, sum_time/requests)

if __name__ == '__main__':
    MRWebLogStats2b_V2.run()

In [ ]:
#@title Run the job
!rm -rf results
!python3.12 -m weblog_stats_2b_v2 -r local --output-dir results --cleanup NONE weblog.txt
!cat results/*

In [13]:
#@title Exercise 2c) 10-second inverter index
%%file weblog_stats_2c.py

from mrjob.job import MRJob, MRStep

class MRWebLogStats2c(MRJob):

  def mapper(self, _, line):
      parts = line.strip().split(' ')
      if len(parts) == 6:   # make sure the line is complete
            ip = parts[1]
            url = parts[4]
            time_interval_10s = parts[0][0:18]
            yield "{}-{}".format(time_interval_10s, url), ip

  def combiner(self, interval_key, ips):
      for ip in set(ips):
        yield interval_key, ip

  def reducer(self, interval_key, ips):
    yield interval_key, list(set(ips))

if __name__ == '__main__':
    MRWebLogStats2c.run()

Writing weblog_stats_2c.py


In [14]:
#@title Run the job
!rm -rf results
!python3.12 -m weblog_stats_2c -r local --output-dir results --cleanup NONE weblog.txt
!cat results/*

No configs found; falling back on auto-configuration
No configs specified for local runner
Creating temp directory /tmp/weblog_stats_2c.root.20261013.174453.972464
Running step 1 of 1...
job output is in results
"2016-12-06T08:58:3-/codemove/01IX95N3AFP4"	["120.52.73.98"]
"2016-12-06T08:58:3-/codemove/0GLNQSHCISWJ"	["120.52.73.98"]
"2016-12-06T08:58:3-/codemove/1N80W0N2R36C"	["120.52.73.97"]
"2016-12-06T08:58:3-/codemove/1U6HCG3V2S9D"	["185.28.193.95"]
"2016-12-06T08:58:3-/codemove/2CEBGK8M78Y7"	["192.241.151.220"]
"2016-12-06T08:58:3-/codemove/5Q9SRR2G46PJ"	["120.52.73.98"]
"2016-12-06T08:58:3-/codemove/6GTXIA9YHX09"	["120.52.73.98"]
"2016-12-06T08:58:3-/codemove/7HIW17K7FDZI"	["97.77.104.22"]
"2016-12-06T08:58:3-/codemove/B35MFVKMU1C4"	["2002:894a:3a93:d:250:56ff:fe00:88c0"]
"2016-12-06T08:58:3-/codemove/BJJHJB8J8T7C"	["120.52.73.97"]
"2016-12-06T08:58:3-/codemove/BRPB8Y32OAGA"	["120.52.73.97"]
"2016-12-06T08:58:3-/codemove/CNDHDS3RE0QZ"	["202.47.236.252"]
"2016-12-06T08:58:3-/codemo